# Analyze Intrinsic Metrics

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../../scripts/")
from analysis.model_info import MODEL_INFORMATION
%load_ext autoreload
%autoreload 2

In [2]:
df = pd.read_json(
    "data/compiled_intrinsic_metrics_ar-cs-de.jsonl",
    orient="records",
    lines=True,
)

In [ ]:
df

In [4]:
import numpy as np
from sklearn.preprocessing import StandardScaler


def calculate_quality_score(df):
    """Calculate an aggregate quality score from intrinsic metrics using z-score normalization.

    Each metric is standardized (mean=0, std=1) and then averaged. This approach:
    - Handles extreme outliers automatically
    - Gives equal weight to each metric in terms of variance
    - Incorporates all 4 metrics: prompts_distinct_ri, responses_distinct_ri,
      rubric_score, and perplexity (log-transformed and inverted)

    df (pd.DataFrame): DataFrame with columns: prompts_distinct_ri,
        responses_distinct_ri, rubric_score, perplexity

    RETURNS (pd.Series): Quality scores for each row (higher is better)
    """
    data = np.column_stack(
        [
            df["prompts_distinct_ri"],
            df["responses_distinct_ri"],
            df["rubric_score"],
            -np.log1p(df["perplexity"]),
        ]
    )

    # Standardize and average
    scaler = StandardScaler()
    normalized = scaler.fit_transform(data)
    quality_score = normalized.mean(axis=1)

    return quality_score

In [ ]:
# Calculate quality score
df["quality_score"] = calculate_quality_score(df)

# Show top models by quality score
df[
    [
        "language",
        "model",
        "quality_score",
        "rubric_score",
        "responses_distinct_ri",
        "prompts_distinct_ri",
        "perplexity",
    ]
].sort_values("quality_score", ascending=False)